In [ ]:
import os
import ast
import torchaudio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import IPython.display as ipd
import soundfile as sf
import librosa
import librosa.display
import plotly.express as px
from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score, f1_score

# **EDA**

## **General Overview and Class & Sample Count Verification** 

In [ ]:
# Paths 
DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
AUDIO_DIR = os.path.join(DATA_DIR, "train_audio")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TAXONOMY_CSV = os.path.join(DATA_DIR, "taxonomy.csv")
SOUNDSCAPES_LABELS_CSV = os.path.join(DATA_DIR, "train_soundscapes_labels.csv")

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df['filepath'] = df['filename'].apply(lambda x: os.path.join(AUDIO_DIR, x))

print(f"Total number of records: {len(df)}")
print(f"Number of unique classes: {df['primary_label'].nunique()}")

The `train.csv` dataset contains 206 classes, but the competition requires predicting 234 classes (based on `taxonomy.csv`). Actions to take:
* Hardcode the model's output layer to exactly 234 classes to prevent Kaggle submission errors.
* Extract training samples for the missing 28 classes from the train_soundscapes_labels.csv file.

Навчальний датасет містить 206 класів, але змагання вимагає передбачення 234 класів (згідно з `taxonomy.csv`). Необхідні дії:
* Зафіксувати вихідний шар моделі на 234 класи, щоб уникнути помилок під час сабміту на Kaggle.
* Витягти навчальні семпли для 28 відсутніх класів із файлу train_soundscapes_labels.csv.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
soundscapes_df = pd.read_csv(SOUNDSCAPES_LABELS_CSV)

train_df['filepath'] = train_df['filename'].apply(lambda x: os.path.join(AUDIO_DIR, x))

all_classes = set(taxonomy_df['primary_label'].unique())
train_classes = set(train_df['primary_label'].unique())
missing_classes = all_classes - train_classes

print(f"Total number of records in train_audio: {len(train_df)}")
print(f"Classes in taxonomy: {len(all_classes)}")
print(f"Classes in train_audio: {len(train_classes)}")
print(f"Missing classes in train_audio: {len(missing_classes)}\n")

# Check if missing classes are in soundscapes
soundscape_classes = set()
for labels_str in soundscapes_df['primary_label'].dropna():
    for label in str(labels_str).split(';'):
        soundscape_classes.add(label.strip())

found_in_soundscapes = missing_classes.intersection(soundscape_classes)
still_missing = missing_classes - soundscape_classes

print(f"Missing classes found in soundscapes: {len(found_in_soundscapes)} out of {len(missing_classes)}")
if still_missing:
    print(f"WARNING! These classes ({len(still_missing)}) are completely missing: {still_missing}")
else:
    print("All missing classes are present in train_soundscapes_labels.csv!")

The primary train_audio dataset contains only 206 out of the 234 target classes. However, all 28 missing classes are successfully located within the `train_soundscapes_labels.csv` dataset.

Основний датасет train_audio містить лише 206 із 234 цільових класів. Проте всі 28 відсутніх класів успішно знайдені у файлі `train_soundscapes_labels.csv`.

In [ ]:
# 1. Get filenames from CSV and actual files from the directory
csv_files = set(pd.read_csv(TRAIN_CSV)['filename'])
folder_files = {
    os.path.relpath(os.path.join(r, f), AUDIO_DIR).replace('\\', '/')
    for r, _, fs in os.walk(AUDIO_DIR) for f in fs if f.endswith(('.ogg', '.wav'))
}

print(f"Unique files in CSV: {len(csv_files)} | Files in folder: {len(folder_files)}\n")

# 2. Check for mismatches
missing_in_folder = csv_files - folder_files
missing_in_csv = folder_files - csv_files

if not missing_in_folder and not missing_in_csv:
    print("✅ All files and names match perfectly!")
else:
    print("❌ Discrepancies found:\n")
    if missing_in_folder:
        print(f"In CSV, but missing in folder ({len(missing_in_folder)} files). Examples: {list(missing_in_folder)[:5]}")
    if missing_in_csv:
        print(f"In folder, but missing in CSV ({len(missing_in_csv)} files). Examples: {list(missing_in_csv)[:5]}")

The metadata perfectly matches the directory contents, confirming that all 35,549 audio files are correctly accounted for and ready for training.

Метадані ідеально збігаються з вмістом директорії, підтверджуючи, що всі 35 549 аудіофайлів на місці та готові до використання у навчанні.

In [ ]:
# 1. Складаємо список усіх унікальних птахів, які є ОСНОВНИМИ (primary_label)
primary_birds = set(train_df['primary_label'].unique())

# 2. Витягуємо всі птахи, які згадуються як ДРУГОРЯДНІ (secondary_labels)
all_secondary_list = []
for labels_str in train_df['secondary_labels']:
    # Перетворюємо рядок "['bird1']" на справжній список []
    labels_list = ast.literal_eval(labels_str)
    all_secondary_list.extend(labels_list)

secondary_birds = set(all_secondary_list)

# 3. Знаходимо тих, хто є ТІЛЬКИ у вторинних мітках
ghost_birds = secondary_birds - primary_birds

print(f"Знайдено птахів, які ні разу не були основними: {len(ghost_birds)}")
if len(ghost_birds) > 0:
    print(f"Ось деякі з них: {list(ghost_birds)[:10]}")
    
    # Подивимось, як часто вони зустрічаються
    counts = Counter(all_secondary_list)
    print("\nТоп-5 'привидів' за кількістю згадок:")
    for bird in list(ghost_birds)[:5]:
        print(f"{bird}: {counts[bird]} разів")

The cross-check confirmed that 28 bird species are entirely absent from the provided train.csv (neither primary nor secondary labels).

Перехресна перевірка підтвердила, що 28 видів птахів повністю відсутні у наданому train.csv (їх немає ні в основних, ні у другорядних мітках).

## **Class Imbalance and Recording Duration Analysis**

In [ ]:
# Get class counts
class_counts = train_df['primary_label'].value_counts()

# Print statistics
print("--- Class Distribution Statistics ---")
print(f"Max samples for a single class: {class_counts.max()} ({class_counts.index[0]})")
print(f"Min samples for a single class: {class_counts.min()} ({class_counts.index[-1]})")
print(f"Median samples per class: {class_counts.median()}")
print(f"Mean samples per class: {class_counts.mean():.2f}\n")

# Plot Top 30 and Bottom 30 classes
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Top 30
class_counts.head(30).plot(kind='bar', ax=ax[0], color='skyblue')
ax[0].set_title('Top 30 Most Frequent Classes')
ax[0].set_ylabel('Number of Samples')

# Bottom 30
class_counts.tail(30).plot(kind='bar', ax=ax[1], color='salmon')
ax[1].set_title('Bottom 30 Least Frequent Classes')
ax[1].set_ylabel('Number of Samples')

plt.tight_layout()
plt.show()

The dataset exhibits a severe class imbalance, with the number of samples ranging drastically from just 1 to 499 per class. This requires implementing balancing strategies like oversampling or weighted loss functions during training to prevent the model from ignoring rare species.

Датасет має критичний дисбаланс класів, де кількість записів варіюється від всього 1 до 499 на клас. Щоб модель не ігнорувала рідкісні види, під час навчання необхідно буде обов'язково застосувати стратегії балансування, такі як оверсемплінг або зважені функції втрат.

In [ ]:
# Function to get duration using soundfile (much more stable for .ogg in Kaggle)
def get_audio_duration(filepath):
    try:
        return sf.info(filepath).duration
    except Exception:
        return np.nan

print("Calculating audio durations for all files using soundfile...")
train_df['duration'] = [get_audio_duration(path) for path in tqdm(train_df['filepath'])]

# Drop any files that failed to load
valid_durations = train_df['duration'].dropna()

# Print statistics
print("\n--- Audio Duration Statistics (seconds) ---")
print(f"Minimum duration: {valid_durations.min():.2f}")
print(f"Maximum duration: {valid_durations.max():.2f}")
print(f"Mean duration: {valid_durations.mean():.2f}")
print(f"Median duration: {valid_durations.median():.2f}")

typical_durations = valid_durations[valid_durations <= 120]

plt.figure(figsize=(12, 6))
# Using 60 bins means each bin represents exactly 2 seconds
plt.hist(typical_durations, bins=60, color='mediumpurple', edgecolor='black', alpha=0.7)

# Add vertical lines for overall Mean and Median
plt.axvline(valid_durations.mean(), color='red', linestyle='dashed', linewidth=2, label=f'Mean: {valid_durations.mean():.1f}s')
plt.axvline(valid_durations.median(), color='green', linestyle='dashed', linewidth=2, label=f'Median: {valid_durations.median():.1f}s')

plt.title("Distribution of Audio Durations (Files ≤ 120 seconds)", fontsize=14)
plt.xlabel("Duration (seconds)", fontsize=12)
plt.ylabel("Number of Audio Files", fontsize=12)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Print what percentage of the dataset this graph represents
percent_under_120 = (len(typical_durations) / len(valid_durations)) * 100
print(f"Percentage of files 120 seconds or shorter: {percent_under_120:.2f}%")

Over 96% of the audio files are under 2 minutes, with a median duration of 21 seconds, though extreme outliers (up to nearly 2 hours) exist.

Понад 96% записів тривають менше 2 хвилин із медіаною у 21 секунду, проте в даних є екстремальні викиди (до 2 годин).

In [ ]:
# Calculate total duration per class in minutes
# Assuming 'duration' column is already computed in train_df from the previous step
class_duration_mins = (train_df.groupby('primary_label')['duration'].sum() / 60).sort_values(ascending=False)

print("--- Total Duration Statistics per Class (Minutes) ---")
print(f"Max duration for a single class: {class_duration_mins.max():.2f} mins ({class_duration_mins.index[0]})")
print(f"Min duration for a single class: {class_duration_mins.min():.2f} mins ({class_duration_mins.index[-1]})")
print(f"Median duration per class: {class_duration_mins.median():.2f} mins\n")

# Plot Top 30 and Bottom 30 by Total Duration
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Top 30
class_duration_mins.head(30).plot(kind='bar', ax=ax[0], color='teal')
ax[0].set_title('Top 30 Classes by Total Duration')
ax[0].set_ylabel('Total Duration (Minutes)')

# Bottom 30
class_duration_mins.tail(30).plot(kind='bar', ax=ax[1], color='coral')
ax[1].set_title('Bottom 30 Classes by Total Duration')
ax[1].set_ylabel('Total Duration (Minutes)')

plt.tight_layout()
plt.show()

Measuring by total duration reveals the true severity of the class imbalance: the most frequent class (`coffal1`) has nearly 10 hours of audio, while the rarest minority classes have less than a minute in total. This massive disparity confirms that relying solely on the number of files is misleading; we must implement strategic chunking, oversampling, and aggressive data augmentation for the underrepresented classes to ensure the model learns their acoustic features properly.

Оцінка за загальною тривалістю показує справжню критичність дисбалансу: найпопулярніший клас (`coffal1`) має майже 10 годин аудіо, тоді як найрідкіснішим видам доступно загалом менше хвилини. Ця величезна прірва підтверджує, що орієнтуватися лише на кількість файлів оманливо; для успішного навчання моделі нам необхідно застосувати стратегічне нарізання (chunking), оверсемплінг та агресивну аугментацію для малопредставлених класів.

## **Sample Rate Distribution Verification**

In [ ]:
# Function to extract sample rate using soundfile
def get_sample_rate(filepath):
    try:
        return sf.info(filepath).samplerate
    except Exception:
        return np.nan

print("Checking sample rates for all files...")
# We use the filepath column created in previous steps
train_df['sample_rate'] = [get_sample_rate(path) for path in tqdm(train_df['filepath'])]

# Drop any files that failed to load
valid_sample_rates = train_df['sample_rate'].dropna()

# Count the occurrences of each sample rate
sr_counts = valid_sample_rates.value_counts()

print("\n--- Sample Rate Distribution ---")
for sr, count in sr_counts.items():
    print(f"{int(sr)} Hz: {count} files ({(count/len(valid_sample_rates))*100:.2f}%)")

# Plot the distribution
plt.figure(figsize=(8, 5))
sr_counts.plot(kind='bar', color='lightseagreen', edgecolor='black', alpha=0.8)
plt.title("Distribution of Sample Rates in train_audio", fontsize=14)
plt.xlabel("Sample Rate (Hz)", fontsize=12)
plt.ylabel("Number of Audio Files", fontsize=12)

# Rotate x-axis labels to be horizontal for better readability
plt.xticks(rotation=0) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Quick check to alert if there are unexpected sample rates
expected_sr = 32000
if len(sr_counts) == 1 and expected_sr in sr_counts:
    print(f"\n✅ SUCCESS: All files are exactly {expected_sr} Hz as expected!")
else:
    print(f"\n⚠️ WARNING: Found multiple sample rates or rates other than {expected_sr} Hz!")

The analysis confirms that 100% of the audio files are uniformly sampled at 32,000 Hz, exactly as stated by the competition organizers.

Аналіз підтверджує, що абсолютно всі аудіофайли мають однакову частоту дискретизації 32 000 Гц, як і було заявлено організаторами змагання.

## **Distribution of Audio Quality Ratings**

In [ ]:
# Check if 'rating' column exists
if 'rating' in train_df.columns:
    print("--- Recording Quality (Rating) Distribution ---")
    
    # Count occurrences of each rating and sort by the rating value
    rating_counts = train_df['rating'].value_counts().sort_index()
    total_rated = len(train_df)
    
    # Print detailed statistics
    for rating, count in rating_counts.items():
        print(f"Rating {rating}: {count} files ({(count/total_rated)*100:.2f}%)")
        
    # Plot the distribution
    plt.figure(figsize=(10, 5))
    rating_counts.plot(kind='bar', color='goldenrod', edgecolor='black', alpha=0.8)
    
    plt.title("Distribution of Audio Quality Ratings", fontsize=14)
    plt.xlabel("Rating (0.0 = Unrated, 1.0 = Low, 5.0 = High)", fontsize=12)
    plt.ylabel("Number of Audio Files", fontsize=12)
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("WARNING: 'rating' column not found in the dataset.")

Over 36% of the dataset is unrated (0.0), which is expected since sources like iNaturalist do not provide user ratings. Among the rated files, the vast majority are of high quality (scores 3.0 to 5.0), with 4.0 and 5.0 being the most frequent. You can also notice fractional ratings (such as 3.5 or 4.5) — this happens because the presence of background species in the recording automatically reduces the base score by 0.5. Extremely low-quality recordings make up less than 4% of the total data.

Понад 36% даних не мають рейтингу (0.0) — це очікувано, оскільки джерела на кшталт iNaturalist не підтримують систему оцінювання. Серед оцінених файлів переважна більшість має високу якість (від 3.0 до 5.0), причому оцінки 4.0 та 5.0 зустрічаються найчастіше. Також помітні дробові значення рейтингу (наприклад, 3.5 або 4.5) — це пояснюється тим, що наявність фонових птахів (secondary labels) на записі автоматично знижує базовий бал на 0.5. Відверто низькоякісні записи становлять менше ніж 4% від загальної кількості.

## **Secondary Birds Analysis**

In [ ]:
# 1. Parse the secondary_labels column to count the number of background birds
# ast.literal_eval safely evaluates a string containing a Python list (e.g., "['bird1', 'bird2']")
train_df['num_secondary'] = train_df['secondary_labels'].apply(
    lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0
)

# 2. Create the filtered dataset (Rating >= 3.0 OR Rating == 0.0)
filtered_df = train_df[(train_df['rating'] >= 3.0) | (train_df['rating'] == 0.0)]

# 3. Calculate distributions
all_data_counts = train_df['num_secondary'].value_counts().sort_index()
filtered_data_counts = filtered_df['num_secondary'].value_counts().sort_index()

# Print Statistics
print("--- Secondary Birds: ALL DATA ---")
for num, count in all_data_counts.items():
    print(f"{num} background bird(s): {count} files ({(count/len(train_df))*100:.2f}%)")

print(f"\nTotal files in filtered dataset: {len(filtered_df)} out of {len(train_df)}")
print("--- Secondary Birds: FILTERED DATA (Rating >= 3.0 or 0.0) ---")
for num, count in filtered_data_counts.items():
    print(f"{num} background bird(s): {count} files ({(count/len(filtered_df))*100:.2f}%)")

# 4. Plotting the results
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Plot for ALL DATA
all_data_counts.plot(kind='bar', ax=ax[0], color='cornflowerblue', edgecolor='black', alpha=0.8)
ax[0].set_title('Number of Secondary Birds (All Data)', fontsize=14)
ax[0].set_xlabel('Number of Background Species', fontsize=12)
ax[0].set_ylabel('Number of Audio Files', fontsize=12)
ax[0].tick_params(axis='x', rotation=0)
ax[0].grid(axis='y', linestyle='--', alpha=0.7)

# Plot for FILTERED DATA
filtered_data_counts.plot(kind='bar', ax=ax[1], color='mediumseagreen', edgecolor='black', alpha=0.8)
ax[1].set_title('Number of Secondary Birds (Rating >= 3.0 or 0.0)', fontsize=14)
ax[1].set_xlabel('Number of Background Species', fontsize=12)
ax[1].set_ylabel('Number of Audio Files', fontsize=12)
ax[1].tick_params(axis='x', rotation=0)
ax[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

Almost 90% of the recordings contain only the primary bird, and filtering out low-quality files barely alters this distribution. Since the test soundscapes will feature noisy environments with multiple overlapping bird calls, a model trained strictly on these isolated vocals might struggle to generalize.

Майже 90% записів містять лише основного птаха, і фільтрація низькоякісних файлів майже не змінює цей розподіл. Оскільки тестові саундскейпи міститимуть шум та одночасний спів багатьох видів, моделі буде важко адаптуватися, якщо навчати її виключно на цих ізольованих "сольних" аудіо.

## **Waveforms and Mel Spectrograms**

In [ ]:
print("--- Visualizing Waveforms and Mel Spectrograms ---")

# Select 3 random files for visualization
sample_files = train_df.sample(3, random_state=42)

for idx, row in sample_files.iterrows():
    filepath = row['filepath']
    label = row['primary_label']
    
    # Load audio using soundfile (fast and reliable)
    y, sr = sf.read(filepath)
    
    # Create subplots: 1 row, 2 columns
    fig, ax = plt.subplots(1, 2, figsize=(16, 4))
    
    # 1. Plot Waveform (Amplitude over Time)
    time_axis = np.linspace(0, len(y)/sr, num=len(y))
    ax[0].plot(time_axis, y, color='royalblue', alpha=0.7)
    ax[0].set_title(f"Waveform: {label} (Duration: {len(y)/sr:.1f}s)", fontsize=12)
    ax[0].set_xlabel("Time (s)")
    ax[0].set_ylabel("Amplitude")
    ax[0].grid(True, alpha=0.3)
    
    # 2. Plot Mel Spectrogram (Frequency over Time)
    # We use typical parameters for bird calls (fmax=14000)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmin=50, fmax=14000)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    img = librosa.display.specshow(
        mel_spec_db, x_axis='time', y_axis='mel', 
        sr=sr, fmax=14000, ax=ax[1], cmap='magma'
    )
    fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
    ax[1].set_title(f"Mel Spectrogram: {label}", fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Display an interactive audio player directly in the notebook
    display(ipd.Audio(data=y, rate=sr))

Listening to the audio reveals critical nuances that spectrograms alone can obscure. For instance, a quiet bird call can be overshadowed by a loud wind artifact (as seen in saytan1), or a loud, broad-frequency bird call might visually resemble background noise (purjay1). This highlights a severe issue with varying Signal-to-Noise Ratios (SNR) and weak labels. If we blindly crop segments, the model might learn to associate a loud gust of wind with a specific bird class, ignoring the actual quiet vocalization.

Прослуховування аудіо розкриває критичні нюанси, які самі лише спектрограми можуть приховувати. Наприклад, тихий спів птаха може перекриватися гучним артефактом вітру (як у saytan1), або ж гучний птах із широким діапазоном частот може візуально нагадувати шум (purjay1). Це підкреслює серйозну проблему з різним співвідношенням сигнал/шум (SNR) та слабкою розміткою. Якщо ми будемо сліпо нарізати фрагменти, модель може навчитися асоціювати гучний порив вітру з певним класом птаха, ігноруючи справжній тихий спів.

## **Geospatial Analysis**

In [ ]:
# Check if geographical data is available
if 'latitude' in train_df.columns and 'longitude' in train_df.columns:
    print("--- Geospatial Distribution of Bird Recordings ---")
    
    # Drop rows where coordinates might be missing
    geo_df = train_df.dropna(subset=['latitude', 'longitude']).copy()
    
    # Create an interactive scatter map
    fig = px.scatter_geo(
        geo_df,
        lat='latitude',
        lon='longitude',
        color='primary_label',  # Different color for each bird
        hover_name='primary_label', # Show bird name on hover
        opacity=0.6,
        title='Global Distribution of BirdCLEF Training Data',
        projection='natural earth' # Makes a nice rounded map
    )
    
    # Hide the legend because displaying 200+ classes will clutter the screen
    fig.update_layout(
        showlegend=False,
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()
else:
    print("WARNING: 'latitude' or 'longitude' columns not found in the dataset.")

The map reveals a strong geographic bias in the training data, with massive concentrations of recordings in the Americas and Europe, while regions like Africa, Asia, and Australia are heavily underrepresented. This spatial clustering means the model will mostly learn the acoustic background noises (wind, insects, other regional fauna) specific to these heavily sampled continents. We can heavily leverage this geospatial data in our model. By feeding longitude and latitude as auxiliary metadata features alongside the audio, the model can learn to filter out species that are geographically impossible for a given location, significantly reducing false positives.

Карта демонструє сильне географічне упередження (bias) у навчальних даних: наявне величезне скупчення записів в Північній/Південній Америці та Європі, тоді як Африка, Азія та Австралія представлені дуже слабко. Таке просторове кластеризування означає, що модель вивчить переважно акустичні фонові шуми (вітер, комахи, інші регіональні тварини), притаманні саме цим континентам. Ми можемо використати ці геопросторові координати як величезну перевагу. Якщо передавати широту та довготу як додаткові метадані (auxiliary features) разом з аудіо, модель зможе автоматично відкидати види птахів, які географічно не можуть перебувати в цій точці, що суттєво зменшить кількість хибнопозитивних спрацьовувань.

# **Validation**

In [ ]:
# 1. Filter out low-quality recordings (ratings between 0.5 and 2.5 inclusive)
# We keep recordings with rating < 0.5 (which includes 0.0) or rating > 2.5
train_df = train_df[(train_df['rating'] < 0.5) | (train_df['rating'] > 2.5)].copy()
train_df.reset_index(drop=True, inplace=True)

print(f"Dataset size after filtering low-quality audio: {len(train_df)} samples\n")

# 2. Initialize the fold column with a default value of -1
train_df['fold'] = -1

# 3. Set up StratifiedGroupKFold (5 folds is the Kaggle standard)
# shuffle=True mixes groups before splitting, random_state=42 ensures reproducibility
N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 4. Perform the split considering class imbalance and authors (groups)
# y=train_df['primary_label'] maintains bird species proportions (stratification)
# groups=train_df['author'] ensures records from the same author stay in the same fold
for fold, (train_idx, val_idx) in enumerate(sgkf.split(train_df, y=train_df['primary_label'], groups=train_df['author'])):
    train_df.loc[val_idx, 'fold'] = fold

# 5. Verification: Check the distribution of files across folds
print(f"--- Validation Strategy: {N_SPLITS}-Fold Stratified Grouping ---")
fold_counts = train_df.groupby('fold').size()
for fold, count in fold_counts.items():
    print(f"Fold {fold}: {count} samples ({(count/len(train_df))*100:.1f}%)")

# 6. Verification: Check the number of unique classes in each fold
print("\nCheck: Number of unique classes per fold:")
print(train_df.groupby('fold')['primary_label'].nunique())

# Save the filtered metadata with folds for training
train_df.to_csv("train_with_folds_filtered.csv", index=False)
print("\n✅ Success: train_with_folds_filtered.csv successfully created!")

# **Metrics**

The evaluation metric for this contest is a version of macro-averaged ROC-AUC that skips classes that have no true positive labels.

In [ ]:
def calculate_competition_metric(y_true, y_pred):
    """
    Calculates macro-averaged ROC-AUC, skipping classes with no true positives.
    y_true: Ground truth labels (one-hot encoded or binary matrix)
    y_pred: Predicted probabilities
    """
    list_auc = []
    # Loop through each class (bird species)
    for i in range(y_true.shape[1]):
        # Check if the class has at least one positive and one negative sample
        # ROC-AUC is undefined if only one class is present in y_true
        if len(np.unique(y_true[:, i])) > 1:
            score = roc_auc_score(y_true[:, i], y_pred[:, i])
            list_auc.append(score)
            
    # Return the mean of all valid AUC scores
    return np.mean(list_auc) if list_auc else 0

print("Metric function 'calculate_competition_metric' is ready!")

In [ ]:
# ==========================================
# 1. Macro mean Average Precision (mAP)
# ==========================================
def calculate_map(y_true, y_pred):
    """
    Рахує mAP, ігноруючи класи без істинних міток.
    Ідеально для оцінки впевненості моделі (карає за хибні спрацьовування).
    """
    list_ap = []
    for i in range(y_true.shape[1]):
        # Рахуємо тільки для тих класів, де є хоча б один істинно позитивний запис
        if np.sum(y_true[:, i]) > 0:
            score = average_precision_score(y_true[:, i], y_pred[:, i])
            list_ap.append(score)
            
    return np.mean(list_ap) if list_ap else 0


# ==========================================
# 2. Macro F1-Score
# ==========================================
def calculate_macro_f1(y_true, y_pred, threshold=0.5):
    """
    Рахує Macro F1-Score. На відміну від AUC, вимагає чіткого порогу (threshold).
    Показує, наскільки добре модель приймає фінальне рішення ("є птах" / "немає птаха").
    """
    # Перетворюємо ймовірності у чіткі бінарні відповіді (0 або 1)
    y_pred_binary = (y_pred >= threshold).astype(int)
    
    list_f1 = []
    for i in range(y_true.shape[1]):
        if np.sum(y_true[:, i]) > 0:
            # zero_division=0 гарантує, що не буде помилки, якщо модель нічого не передбачила
            score = f1_score(y_true[:, i], y_pred_binary[:, i], zero_division=0)
            list_f1.append(score)
            
    return np.mean(list_f1) if list_f1 else 0


# ==========================================
# 3. Top-K Accuracy (Top-3)
# ==========================================
def calculate_top_k_accuracy(y_true, y_pred, k=3):
    """
    Рахує частку аудіофайлів, де істинний птах опинився у Топ-К (за замовчуванням Топ-3)
    найбільш ймовірних передбачень моделі.
    """
    correct = 0
    total = y_true.shape[0]
    
    for i in range(total):
        # Знаходимо індекси K найбільших ймовірностей для цього аудіо
        top_k_indices = np.argsort(y_pred[i])[-k:]
        
        # Знаходимо індекси реальних птахів (де y_true == 1)
        true_indices = np.where(y_true[i] == 1)[0]
        
        # Якщо хоча б один правильний птах є серед Топ-K передбачень
        if len(set(top_k_indices).intersection(set(true_indices))) > 0:
            correct += 1
            
    return correct / total if total > 0 else 0